In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import json
import yaml
from typing import Literal
from pydantic import BaseModel, Field
from typing_extensions import List

load_dotenv()

True

In [2]:
model = init_chat_model("groq:openai/gpt-oss-120b")

In [3]:
from pathlib import Path

schema_content = Path(
    "ecommerce_analytics\\models\\dbt_mrt\\_dbt_mrt_schema.yml"
).read_text(encoding="utf-8")

In [4]:
class ColumnTest(BaseModel):
    test_type: Literal[
        "not_null", "unique", "accepted_values", "relationships", "other"
    ] = Field(
        description="Type of dbt column test",
    )
    accepted_values: List[str] | None = Field(
        default=None,
        description="Accepted values when test_type is accepted_values",
    )
    reference_table: str | None = Field(
        default=None,
        description="Referenced table when test_type is relationships",
    )
    reference_column: str | None = Field(
        default=None,
        description="Referenced column when test_type is relationships",
    )

In [5]:
class CanonicalColumn(BaseModel):
    column_name: str = Field(description="Name of the column")
    description: str | None = Field(
        default=None,
        description="Column description copied from source metadata when available",
    )
    possible_synonyms: List[str] | None = Field(
        default=None,
        description="Possible business synonyms users might use for this column",
    )
    tests: List[ColumnTest] | None = Field(
        default=None,
        description="dbt tests defined for this column",
    )

In [6]:
class CanonicalTable(BaseModel):
    table_name: str = Field(description="Canonical table name")
    description: str | None = Field(
        default=None,
        description="Table description copied from source metadata when available",
    )
    grain: str | None = Field(
        default=None,
        description="Table grain when explicitly present in source metadata",
    )
    primary_key: List[str] | None = Field(
        default=None,
        description="Primary key columns explicitly supported by source metadata",
    )
    columns: List[CanonicalColumn] = Field(
        default_factory=list,
        description="Columns in the canonical table",
    )

In [7]:
class ForeignKeyRelationship(BaseModel):
    source_table: str = Field(description="Table containing the foreign key")
    source_column: str = Field(description="Foreign key column in the source table")
    target_table: str = Field(description="Referenced table")
    target_column: str = Field(description="Referenced column in the target table")

In [8]:
class CanonicalSchema(BaseModel):
    tables: List[CanonicalTable] = Field(
        default_factory=list,
        description="Canonical tables available to the text-to-sql agent",
    )
    relationships: List[ForeignKeyRelationship] = Field(
        default_factory=list,
        description="Explicit foreign key relationships between tables",
    )

In [9]:
schema_yaml = yaml.safe_load(schema_content)
model_entries = schema_yaml["models"]

# Enrich exploration schema with the review embeddings side table
# created in embeddings/create_review_embeddings.py.
embedding_model_name = "dbt_mrt_review_embeddings"
embedding_model_exists = any(
    model_entry.get("name") == embedding_model_name for model_entry in model_entries
)

if not embedding_model_exists:
    model_entries.append(
        {
            "name": embedding_model_name,
            "description": (
                "Side table containing vector embeddings for review_combined_text "
                "from dbt_mrt_reviews."
            ),
            "columns": [
                {
                    "name": "order_id",
                    "description": "Order identifier copied from dbt_mrt_reviews.",
                    "tests": ["not_null", "unique"],
                },
                {
                    "name": "review_combined_text_embedding",
                    "description": (
                        "1024-dimensional vector embedding generated from "
                        "review_combined_text."
                    ),
                },
            ],
        }
    )

In [10]:
class CanonicalRelationshipDigest(BaseModel):
    relationships: List[ForeignKeyRelationship] = Field(
        default_factory=list,
        description="Explicit foreign key relationships between tables",
    )

In [11]:
system_prompt_canonical_table_small = """
You are extracting one CanonicalTable object from one dbt model definition.

Return exactly one CanonicalTable object.

Rules:
- Use only information supported by the YAML.
- Do not invent columns, tests, primary keys, or grain.
- If a field is missing or not strongly supported, return null or an empty list.
- Include 0 to 2 conservative possible_synonyms per column.
- Map dbt tests into ColumnTest objects.
- For primary_key, include columns only when explicitly supported by tests or clear primary-key wording.
- For grain, include it only when explicitly stated or strongly supported by the model description.
- For tests:
  - map `not_null` to test_type `not_null`
  - map `unique` to test_type `unique`
  - map `accepted_values` to test_type `accepted_values` and fill `accepted_values`
  - map `relationships` to test_type `relationships` and fill `reference_table` and `reference_column`
  - anything else becomes test_type `other`
- Do not create fields that are not defined in the schema.
- Keep wording short and precise.
""".strip()


system_prompt_canonical_relationships_small = """
You are extracting explicit foreign key relationships from a dbt schema YAML file and table summaries.

Return exactly one CanonicalRelationshipDigest object.

Rules:
- Include only relationships explicitly supported by dbt relationship tests.
- Do not invent relationships.
- Use only the fields defined in ForeignKeyRelationship.
- Prefer an empty list over guessed output.
""".strip()

In [12]:
canonical_table_llm = model.with_structured_output(CanonicalTable)
canonical_relationship_llm = model.with_structured_output(CanonicalRelationshipDigest)

In [13]:
canonical_tables = []

for model_entry in model_entries:
    model_yaml = yaml.safe_dump(
        {"model": model_entry}, sort_keys=False, allow_unicode=False
    )
    table_schema = canonical_table_llm.invoke(
        [
            ("system", system_prompt_canonical_table_small),
            (
                "user",
                "Extract one CanonicalTable object from this dbt model.\n\n```yaml\n"
                + model_yaml
                + "\n```",
            ),
        ]
    )
    canonical_tables.append(table_schema)

relationship_context = {
    "source_yaml": schema_yaml,
    "tables": [table.model_dump() for table in canonical_tables],
}

relationship_digest = canonical_relationship_llm.invoke(
    [
        ("system", system_prompt_canonical_relationships_small),
        (
            "user",
            "Extract explicit foreign key relationships from the dbt schema YAML and table summaries below.\n\n"
            + json.dumps(relationship_context, indent=2),
        ),
    ]
)

schema = CanonicalSchema(
    tables=canonical_tables,
    relationships=relationship_digest.relationships,
)

schema

CanonicalSchema(tables=[CanonicalTable(table_name='dbt_mrt_customers', description='Final customer dimension table at the customer_unique_id grain. Provides a holistic view of customer behavior, including lifetime value (LTV), loyalty status, and geographic patterns.', grain='customer_unique_id', primary_key=['customer_unique_id'], columns=[CanonicalColumn(column_name='customer_unique_id', description='Primary Key. Global unique identifier for the human customer.', possible_synonyms=['customer_id', 'unique_customer_id'], tests=[ColumnTest(test_type='unique', accepted_values=None, reference_table=None, reference_column=None), ColumnTest(test_type='not_null', accepted_values=None, reference_table=None, reference_column=None)]), CanonicalColumn(column_name='customer_city', description=None, possible_synonyms=None, tests=[ColumnTest(test_type='not_null', accepted_values=None, reference_table=None, reference_column=None)]), CanonicalColumn(column_name='customer_state', description=None, pos

In [14]:
schema_dict = schema.model_dump()

Path("schema_digest.json").write_text(
    json.dumps(schema_dict, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

Path("schema_digest.yaml").write_text(
    yaml.safe_dump(schema_dict, sort_keys=False, allow_unicode=True),
    encoding="utf-8",
)

6936

In [15]:
json_path = Path("schema_digest.json").resolve()
yaml_path = Path("schema_digest.yaml").resolve()

print(f"JSON saved to: {json_path}")
print(f"YAML saved to: {yaml_path}")

JSON saved to: D:\Projects\text-to-sql-agent\schema_digest.json
YAML saved to: D:\Projects\text-to-sql-agent\schema_digest.yaml


In [16]:
print(Path("schema_digest.json").read_text(encoding="utf-8")[:1000])
print("\n--- YAML preview ---\n")
print(Path("schema_digest.yaml").read_text(encoding="utf-8")[:1000])

{
  "tables": [
    {
      "table_name": "dbt_mrt_customers",
      "description": "Final customer dimension table at the customer_unique_id grain. Provides a holistic view of customer behavior, including lifetime value (LTV), loyalty status, and geographic patterns.",
      "grain": "customer_unique_id",
      "primary_key": [
        "customer_unique_id"
      ],
      "columns": [
        {
          "column_name": "customer_unique_id",
          "description": "Primary Key. Global unique identifier for the human customer.",
          "possible_synonyms": [
            "customer_id",
            "unique_customer_id"
          ],
          "tests": [
            {
              "test_type": "unique",
              "accepted_values": null,
              "reference_table": null,
              "reference_column": null
            },
            {
              "test_type": "not_null",
              "accepted_values": null,
              "reference_table": null,
              "reference